# Circumplex: analysis and figures

Regenerates every README figure and statistic from `runs/` (unzip
`circumplex_results.zip` from Kaggle at the repo root first). Runs on CPU in a few
minutes. Sections whose inputs are missing are skipped and listed under `status`.

In [ ]:
import json
from pathlib import Path

from IPython.display import HTML, Image, Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
%cd {ROOT}

In [ ]:
!python scripts/run_analysis.py --main-run roberta_full_s42
A = json.loads(Path("runs/analysis/analysis.json").read_text())
A["status"]

## RQ1: models

In [ ]:
p = Path("runs/analysis/results_table.md")
display(Markdown(p.read_text() if p.exists() else "_no runs yet_"))
for t in A.get("pairwise_tests", []):
    print(f"{t['a']} vs {t['b']}: Δ macro-F1 = {t['delta']:+.4f} [{t['lo']:+.4f}, {t['hi']:+.4f}], permutation p = {t['perm_p_value']:.4g}")
for f in ("class_distribution", "rq1_per_class_f1", "calibration"):
    if Path(f"figures/{f}.png").exists():
        display(Image(f"figures/{f}.png"))

## RQ2: confusion vs valence/arousal distance

In [ ]:
for k in ("rq2_model_vs_distance", "rq2_human_vs_distance", "rq2_model_vs_human"):
    print(k, A.get(k, "TBD"))
for f in ("rq2_model_confusion_vs_distance", "rq2_human_confusion_vs_distance", "rq2_model_confusion", "rq2_human_confusion"):
    if Path(f"figures/{f}.png").exists():
        display(Image(f"figures/{f}.png"))

## RQ3: does the embedding space recover valence/arousal?

In [ ]:
print(json.dumps(A.get("rq3", "TBD"), indent=1, default=str)[:3000])
for f in ("rq3_layerwise_valence", "rq3_layerwise_arousal", "rq3_pca_finetuned", "rq3_pca_base", "rq3_pca_random"):
    if Path(f"figures/{f}.png").exists():
        display(Image(f"figures/{f}.png"))

## RQ4: mixed emotions

In [ ]:
print(A.get("rq4_human_vs_model", "TBD"))
for f in ("rq4_human_npmi", "rq4_model_npmi", "rq4_cardinality"):
    if Path(f"figures/{f}.png").exists():
        display(Image(f"figures/{f}.png"))

## RQ5: transfer to cognitive distortions

In [ ]:
print(A.get("rq5_paired_differences", "TBD"))
if Path("figures/rq5_learning_curves.png").exists():
    display(Image("figures/rq5_learning_curves.png"))

## Token attributions and error analysis

In [ ]:
if Path("figures/attributions.html").exists():
    display(HTML(Path("figures/attributions.html").read_text()))
errs = Path("runs/roberta_full_s42/errors.jsonl")
if errs.exists():
    for line in errs.read_text().splitlines()[:15]:
        e = json.loads(line)
        print(f"{e['wrong_prob']:.2f} said {e['wrong_label']:<14} gold {e['gold']}  | {e['text'][:90]}")